In [ ]:
import sys
import os

# Make config.py 
sys.path.append(os.path.abspath(".."))

import pandas as pd
import config

print("Config loaded.")
print("Clip catalog path:", config.CLIP_CATALOG_PATH)

Config loaded.
Clip catalog path: /Users/qian/KWF/rainforest-audio-detection/data/clip_catalog.csv


In [11]:
# Load the full clip catalog
df = pd.read_csv(config.CLIP_CATALOG_PATH)
print("Full catalog loaded:", df.shape)

# Columns we need for labeling
keep_cols = [
    "clip_name", "Recorder", "Timestamp", "Datetime", "Time Of Day",
    "Sim Type", "species", "confidence","audio_path",
]

# Trimmed working copy
labels = df[keep_cols].copy()
print("Trimmed working copy:", labels.shape)
labels.head()

Full catalog loaded: (631317, 33)
Trimmed working copy: (631317, 9)


,clip_name,Recorder,Timestamp,Datetime,Time Of Day,Sim Type,species,confidence,audio_path
0,Audio_Moth_1_20250317_093112.wav,Audio_Moth_1,20250317_093112,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
1,Audio_Moth_1_20250317_093115.wav,Audio_Moth_1,20250317_093115,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
2,Audio_Moth_1_20250317_093118.wav,Audio_Moth_1,20250317_093118,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
3,Audio_Moth_1_20250317_093121.wav,Audio_Moth_1,20250317_093121,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
4,Audio_Moth_1_20250317_093124.wav,Audio_Moth_1,20250317_093124,2025-03-17 10:00:00,morning,[],[],0.0,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...


In [12]:
# Initialize label columns — everything starts as "unknown"
labels["meaningful"] = "unknown"
labels["meaningful_source"] = "unlabeled"

# Confirm the starting state
print(labels["meaningful"].value_counts())
print()
print(labels["meaningful_source"].value_counts())

meaningful
unknown    631317
Name: count, dtype: int64

meaningful_source
unlabeled    631317
Name: count, dtype: int64


In [13]:
# Human activity carve-out: any clip in a sim event window is meaningful
human_mask = labels["Sim Type"] != "[]"

labels.loc[human_mask, "meaningful"] = "meaningful"
labels.loc[human_mask, "meaningful_source"] = "human_activity"

print(labels["meaningful"].value_counts())
print()
print(labels["meaningful_source"].value_counts())

meaningful
unknown       620446
meaningful     10871
Name: count, dtype: int64

meaningful_source
unlabeled         620446
human_activity     10871
Name: count, dtype: int64


In [14]:
fullpred_path = "/Users/qian/KWF/Team Jacama/Stage1_py/evaluation_results/full_predictions.csv"

fp_df = pd.read_csv(fullpred_path)
print("Shape:", fp_df.shape)
print()
print("Columns:", list(fp_df.columns))
print()
print(fp_df[["clip_name", "model_confidence"]].head())

Shape: (631317, 8)

Columns: ['clip_name', 'Recorder', 'confidence', 'Human Activity Score', 'audio_path', 'ground_truth', 'prediction', 'model_confidence']

                          clip_name  model_confidence
0  Audio_Moth_1_20250317_093112.wav          0.170993
1  Audio_Moth_1_20250317_093115.wav          0.004758
2  Audio_Moth_1_20250317_093118.wav          0.472877
3  Audio_Moth_1_20250317_093121.wav          0.279459
4  Audio_Moth_1_20250317_093124.wav          0.999982


In [15]:
# Join model scores onto labels by clip_name
score_lookup = fp_df[["clip_name", "model_confidence"]]
labels = labels.merge(score_lookup, on="clip_name", how="left")

# Check the join worked — how many labels rows got a score?
print("Total labels rows:", len(labels))
print("Rows with a model score:", labels["model_confidence"].notna().sum())
print("Rows missing a model score:", labels["model_confidence"].isna().sum())

Total labels rows: 631317
Rows with a model score: 631317
Rows missing a model score: 0


In [16]:
# BirdNET named-species carve-out: any clip with a named species is meaningful.
# Only label clips still "unknown" — human_activity (gold standard) keeps its source.
species_mask = (labels["species"] != "[]") & (labels["meaningful"] == "unknown")

labels.loc[species_mask, "meaningful"] = "meaningful"
labels.loc[species_mask, "meaningful_source"] = "birdnet_species"

print(labels["meaningful"].value_counts())
print()
print(labels["meaningful_source"].value_counts())

meaningful
unknown       512377
meaningful    118940
Name: count, dtype: int64

meaningful_source
unlabeled          512377
birdnet_species    108069
human_activity      10871
Name: count, dtype: int64


In [17]:
import os

# Make sure the outputs directory exists, then save
os.makedirs(config.OUTPUTS_DIR, exist_ok=True)
labels.to_csv(config.LABELS_PROGRESS_PATH, index=False)
print("Saved to:", config.LABELS_PROGRESS_PATH)

Saved to: /Users/qian/KWF/rainforest-audio-detection/outputs/labels_progress.csv


In [18]:
# Look at clips that have a named species
has_species = labels[labels["species"] != "[]"]
print("Clips with a named species:", len(has_species))
print()

# Show a few examples of species + confidence together
has_species[["clip_name", "species", "confidence", "audio_path"]].head(10)

Clips with a named species: 108855



,clip_name,species,confidence,audio_path
105,Audio_Moth_1_20250317_093627.wav,['Hylophylax naevioides_Spotted Antbird'],0.341533,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
114,Audio_Moth_1_20250317_093654.wav,['Mionectes olivaceus_Olive-striped Flycatcher'],0.267438,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
128,Audio_Moth_1_20250317_093736.wav,['Mionectes olivaceus_Olive-striped Flycatcher'],0.284766,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
208,Audio_Moth_1_20250317_094136.wav,['Malacoptila panamensis_White-whiskered Puffb...,0.463547,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
347,Audio_Moth_1_20250317_094833.wav,['Thamnophilus bridgesi_Black-hooded Antshrike'],0.560182,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
350,Audio_Moth_1_20250317_094842.wav,['Thamnophilus bridgesi_Black-hooded Antshrike'],0.287037,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
375,Audio_Moth_1_20250317_094957.wav,['Claravis pretiosa_Blue Ground Dove'],0.255881,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
376,Audio_Moth_1_20250317_095000.wav,['Myiothlypis fulvicauda_Buff-rumped Warbler'],0.948536,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
409,Audio_Moth_1_20250317_095139.wav,"[""Trogon bairdii_Baird's Trogon""]",0.259667,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
410,Audio_Moth_1_20250317_095142.wav,"[""Trogon bairdii_Baird's Trogon""]",0.351265,/Users/qian/KWF/Segmented_Foldered/Audio Moth ...
